In [4]:

import os, time, threading
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import scipy.io

os.makedirs("model2", exist_ok=True)

N_EPOCH    = 50
LEARN_RATE = 0.01
L2_FACTOR  = 0.01
BATCH_SIZE = 50
N_CLASSES  = 150
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [3]:
!wget https://public.boxcloud.com/d/1/b1!LyWd1nJLoISKX6lZgXv6sn4XrolooCOkFEstmWV2iwr7-kqrtNX08ccniTSpqZL9PxF9KApqA1lXv3hqNAxaH500e0CPNKYKnp1QSZgJiI4ni5r4KbipAx-MG_Tu76vMgxJahw0ex7Y9dlbLDKN1nmwLxjM5Shnrstni-0NxV9bELMf48UPv3lhweLcEJ6HwjZ8vhTD5rPduMpzIAQrefMTEyT8Qy4TdhayJilCDTPbSmXxsSEdPNGWbgnBmjaAcqr9Yc0nEYmc3Jysk21LWc8c92TFT-Z--4QzTCGvv-6-kiXOjNzJnja0bCk8BF2FDBz4tc3fxvwYNIwD6OO6RnTyF4KEvR2yCC8zKl_M9iL7BOFHeo5Bt4qfWioXNzUtF4Y1Lm6kwP8D91TpdDmlXJ_bFB-izlnTA2G4C-JhrbkAFuWWU1b-z0Oa-y5Q7A0meGt4hPVXz_8gZc4isVtZpRK1E8P7jNZy4BepSqU1RoV7emGtIWgCIXdv6ohUVLMS-GxwLiZdisqqp4hpkpDOeilD6UkjXZGhONF6cjxnVTmag1JI7iBdXq4fbh3CUSZSoq9RfXRMJC7lGWC4dyZGJlPH2jNr6eSm7LejL9SRQHAROJpUMlQDX6ldX1R6PeLndTyQwCb4NiS5KCex-D4037B7YUFSW5Bhv2gzRQ6Yv8K0vk72GAYKc3VV71rK-hkwOTHeD1mblQwrV-j78_0k3a0o-ob6Y0BE6A4C6Io0T_RoIJkFz51_xqvuEoV4NMfE8vMREgn0ZyJGzYGkcanrv4YZEktZq56p-9S3VHo4ka904Oe-AnOPAoUoDk2KPgsrnmmVyQSBgTlEpDfTOc4wTwwZV14kWRL8Lequuq--xvUHVS5OiMFFEjWBX3tTbVxV2YQJwxStx05iJ8S7EzL-OjBYBt2laf5dV0KzJ1UHClpjc8lFE-HXY1Y9O-aqIkSHipCAwtpQNCHJLR94W5TnzcSZZWzootJ63wc6KUgT1w4-17GAatp5PDAUlYRmXwnz5SrhkZih8nwdNzUsW-AIzpBQEZFTlYXptaNh135k4c3Tz6bFBVSnYKDQG58pFrxVw4zcNGCvYi1W20zcQ6jxs_fi71flNFJjtuaI44XNrsybcPyoMsqo1NP-r13gC7MJNyq5IpZO4RwL2wq9b41RA2_3FwlGkCxqbxElPYBsIl3rARWI9gbCODgqd0ctbr4DX-hn3v2913qx5k219nzvDTh0cBZ6vHE3_tj38pIZlO-6mPvw7akjjYW5C__Pto_xeklmPiFXAlxxpBNMZRn9cwjTx0fPB7quZbs3Mi6-3TBTTSJpB6DC3v8ypCtyAwM9CiGKHhPOFCxT3j_B8a5MUIF7QfKs364OZrUd5hJk0_LeTn8SQdwlR8_inRB1kMy--3JGq_l84Lw../download -O dataset_lab_150.mat

--2026-05-21 00:28:26--  https://public.boxcloud.com/d/1/b1!LyWd1nJLoISKX6lZgXv6sn4XrolooCOkFEstmWV2iwr7-kqrtNX08ccniTSpqZL9PxF9KApqA1lXv3hqNAxaH500e0CPNKYKnp1QSZgJiI4ni5r4KbipAx-MG_Tu76vMgxJahw0ex7Y9dlbLDKN1nmwLxjM5Shnrstni-0NxV9bELMf48UPv3lhweLcEJ6HwjZ8vhTD5rPduMpzIAQrefMTEyT8Qy4TdhayJilCDTPbSmXxsSEdPNGWbgnBmjaAcqr9Yc0nEYmc3Jysk21LWc8c92TFT-Z--4QzTCGvv-6-kiXOjNzJnja0bCk8BF2FDBz4tc3fxvwYNIwD6OO6RnTyF4KEvR2yCC8zKl_M9iL7BOFHeo5Bt4qfWioXNzUtF4Y1Lm6kwP8D91TpdDmlXJ_bFB-izlnTA2G4C-JhrbkAFuWWU1b-z0Oa-y5Q7A0meGt4hPVXz_8gZc4isVtZpRK1E8P7jNZy4BepSqU1RoV7emGtIWgCIXdv6ohUVLMS-GxwLiZdisqqp4hpkpDOeilD6UkjXZGhONF6cjxnVTmag1JI7iBdXq4fbh3CUSZSoq9RfXRMJC7lGWC4dyZGJlPH2jNr6eSm7LejL9SRQHAROJpUMlQDX6ldX1R6PeLndTyQwCb4NiS5KCex-D4037B7YUFSW5Bhv2gzRQ6Yv8K0vk72GAYKc3VV71rK-hkwOTHeD1mblQwrV-j78_0k3a0o-ob6Y0BE6A4C6Io0T_RoIJkFz51_xqvuEoV4NMfE8vMREgn0ZyJGzYGkcanrv4YZEktZq56p-9S3VHo4ka904Oe-AnOPAoUoDk2KPgsrnmmVyQSBgTlEpDfTOc4wTwwZV14kWRL8Lequuq--xvUHVS5OiMFFEjWBX3tTbVxV2YQJwxStx05iJ8S7EzL-OjBYBt2laf5dV0KzJ1UHClpjc

In [5]:

print("Loading data …")
data    = scipy.io.loadmat("dataset_lab_150.mat")
csi_raw = np.concatenate([data[f"csi{i}"] for i in range(1, 6)], axis=3)
csi_t   = np.concatenate([np.abs(csi_raw), np.angle(csi_raw)], axis=2)
X_np    = csi_t.transpose(3, 2, 0, 1).astype(np.float32)
_, y_np = np.unique(data["label"].flatten(), return_inverse=True)
y_np    = y_np.astype(np.int64)
user_np = np.repeat(np.arange(1, 6), 1500).astype(np.int64)


Loading data …


In [15]:
data['csi1'].shape

(200, 30, 3, 1500)

In [17]:
class SignFiCNN(nn.Module):
    """
    Paper-aligned 9-layer CNN-style classifier for SignFi.
    Layers counted as in paper-style breakdown:
      input/reshape + (conv+bn+relu+pool)*2 + dropout + fc + softmax(logits via CE)
    """
    def __init__(self, input_shape, num_classes=N_CLASSES, dropout_p=0.6):
        super().__init__()
        c_in, h, w = input_shape

        self.features = nn.Sequential(
            nn.Conv2d(c_in, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16, eps=1e-6),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32, eps=1e-6),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            flat_dim = self.features(torch.zeros(1, c_in, h, w)).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_p),
            nn.Linear(flat_dim, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def worker(job_id, X, y, user_np, results):
    if job_id == 0:
        rng   = np.random.default_rng(42)
        idx   = rng.permutation(len(user_np))
        split = int(0.8 * len(idx))
        tr, te = idx[:split], idx[split:]
        tag  = "random_split"
        path = "model2/random_split.pt"
    else:
        test_user = job_id
        tr = np.where(user_np != test_user)[0]
        te = np.where(user_np == test_user)[0]
        tag  = f"loso_user{test_user}_out"
        path = f"model2/loso_user{test_user}.pt"

    X_tr = X[tr].clone()
    y_tr = y[tr].clone()
    X_te = X[te].clone()
    y_te = y[te].clone()

    mean = X_tr.mean(dim=(0, 2, 3), keepdim=True)
    std  = X_tr.std(dim=(0, 2, 3),  keepdim=True).clamp(min=1e-6)
    X_tr = (X_tr - mean) / std
    X_te = (X_te - mean) / std

    model     = SignFiCNN(tuple(X_tr.shape[1:])).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LEARN_RATE,
                          momentum=0.9, weight_decay=L2_FACTOR)
    loader    = DataLoader(TensorDataset(X_tr, y_tr),
                           batch_size=BATCH_SIZE, shuffle=True,
                           pin_memory=(DEVICE.type == "cuda"))

    t0 = time.time()
    for _ in range(N_EPOCH):
        model.train()
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()

    torch.save(model.state_dict(), path)

    model.eval()
    with torch.no_grad():
        preds = model(X_te.to(DEVICE)).argmax(dim=1).cpu()
    acc = (preds == y_te).float().mean().item()
    print(f"  [{tag:25s}]  acc={acc:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    results[job_id] = (tag, acc)



In [ ]:
print("Loading data …")
data    = scipy.io.loadmat("dataset_lab_150.mat")
csi_raw = np.concatenate([data[f"csi{i}"] for i in range(1, 6)], axis=3)
csi_t   = np.concatenate([np.abs(csi_raw), np.angle(csi_raw)], axis=2)
X_np    = csi_t.transpose(3, 2, 0, 1).astype(np.float32)
_, y_np = np.unique(data["label"].flatten(), return_inverse=True)
y_np    = y_np.astype(np.int64)
user_np = np.repeat(np.arange(1, 6), 1500).astype(np.int64)

X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

print(f"Data: {tuple(X.shape)}  |  device: {DEVICE}")


def train_eval_split(X, y, tr_idx, te_idx, tag):
    tr = np.asarray(tr_idx, dtype=np.int64)
    te = np.asarray(te_idx, dtype=np.int64)

    X_tr = X[tr].clone()
    y_tr = y[tr].clone()
    X_te = X[te].clone()
    y_te = y[te].clone()

    mean = X_tr.mean(dim=(0, 2, 3), keepdim=True)
    std  = X_tr.std(dim=(0, 2, 3), keepdim=True).clamp(min=1e-6)
    X_tr = (X_tr - mean) / std
    X_te = (X_te - mean) / std

    model = SignFiCNN(tuple(X_tr.shape[1:])).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LEARN_RATE,
                          momentum=0.9, weight_decay=L2_FACTOR)
    loader = DataLoader(TensorDataset(X_tr, y_tr),
                        batch_size=BATCH_SIZE, shuffle=True,
                        pin_memory=(DEVICE.type == "cuda"))

    t0 = time.time()
    for _ in range(N_EPOCH):
        model.train()
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_te.to(DEVICE)).argmax(dim=1).cpu()
    acc = (preds == y_te).float().mean().item()
    print(f"  [{tag:30s}]  acc={acc:.4f}  ({time.time()-t0:.0f}s)")
    return acc


def kfold_indices(indices, n_splits=5, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.array(indices, dtype=np.int64)
    idx = rng.permutation(idx)
    folds = np.array_split(idx, n_splits)
    out = []
    for k in range(n_splits):
        te = folds[k]
        tr = np.concatenate([folds[i] for i in range(n_splits) if i != k])
        out.append((tr, te))
    return out


results = {}

# 1) Within-user study (Fig. 17a style): 5-fold CV for each user separately.
print("\nRunning within-user 5-fold CV (per user) …")
within_user_means = []
for user in range(1, 6):
    user_idx = np.where(user_np == user)[0]
    folds = kfold_indices(user_idx, n_splits=5, seed=100 + user)
    fold_accs = []
    for fold_id, (tr, te) in enumerate(folds, start=1):
        tag = f"within_user{user}_fold{fold_id}"
        acc = train_eval_split(X, y, tr, te, tag)
        fold_accs.append(acc)
    mean_acc = float(np.mean(fold_accs))
    within_user_means.append(mean_acc)
    results[f"within_user{user}_5fold_mean"] = mean_acc
    print(f"  [within_user{user}_5fold_mean{'':11s}]  {mean_acc:.4f}")

results["within_user_overall_mean"] = float(np.mean(within_user_means))

# 2) Mixed-user 5-fold CV (Fig. 17c 5-fold setting): mix all users then random 5-fold.
print("\nRunning mixed-user 5-fold CV …")
all_idx = np.arange(len(user_np), dtype=np.int64)
folds = kfold_indices(all_idx, n_splits=5, seed=42)
mixed_fold_accs = []
for fold_id, (tr, te) in enumerate(folds, start=1):
    tag = f"mixed_5fold_fold{fold_id}"
    acc = train_eval_split(X, y, tr, te, tag)
    mixed_fold_accs.append(acc)
results["mixed_5fold_mean"] = float(np.mean(mixed_fold_accs))
print(f"  [mixed_5fold_mean{'':17s}]  {results['mixed_5fold_mean']:.4f}")

# 3) Leave-one-subject-out CV (Fig. 17c LOSO).
print("\nRunning leave-one-subject-out CV …")
loso_accs = []
for test_user in range(1, 6):
    tr = np.where(user_np != test_user)[0]
    te = np.where(user_np == test_user)[0]
    tag = f"loso_user{test_user}_out"
    acc = train_eval_split(X, y, tr, te, tag)
    loso_accs.append(acc)
    results[tag] = acc
results["loso_mean"] = float(np.mean(loso_accs))

print("\n" + "=" * 62)
print("SUMMARY")
print("=" * 62)
for user in range(1, 6):
    print(f"within_user{user}_5fold_mean: {results[f'within_user{user}_5fold_mean']:.4f}")
print(f"within_user_overall_mean: {results['within_user_overall_mean']:.4f}")
print(f"mixed_5fold_mean:         {results['mixed_5fold_mean']:.4f}")
for test_user in range(1, 6):
    print(f"loso_user{test_user}_out:       {results[f'loso_user{test_user}_out']:.4f}")
print(f"loso_mean:               {results['loso_mean']:.4f}")


